# STRAT-002 v5 - VectorBT Backtest (Fixed)

Using VectorBT's trailing stop with correct API.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import vectorbt as vbt
import warnings
warnings.filterwarnings('ignore')

print(f"VectorBT version: {vbt.__version__}")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
df = df.sort_index()

# Create RL z-score
df['rl_zscore'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()

df = df[df.index >= '2018-12-15'].dropna()
print(f"Data: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")

In [ ]:
# Entry signals
entry_condition = (
    (df['sopr'] < 1) & 
    (df['sopr_sth'] < 1) & 
    (df['rl_zscore'] > 0.5)
)
entries = entry_condition & ~entry_condition.shift(1).fillna(False)

print(f"Entry signals: {entries.sum()}")

In [ ]:
# VectorBT Portfolio with trailing stop
# Use sl_stop with sl_trail=True for trailing stop
close = df['price']

pf = vbt.Portfolio.from_signals(
    close=close,
    entries=entries,
    exits=None,
    sl_stop=0.30,      # 30% stop
    sl_trail=True,     # Make it trailing
    stop_exit_price='close',
    fees=0.001,
    init_cash=100000,
    freq='D'
)

print("VECTORBT STATS")
print("="*60)
print(pf.stats())

In [ ]:
# Trade details
print("\nTRADES")
print("="*100)
trades_df = pf.trades.records_readable
print(trades_df.to_string())

In [ ]:
# Key metrics
print("\n" + "="*60)
print("KEY METRICS")
print("="*60)
print(f"Total Return: {pf.total_return() * 100:+,.0f}%")
print(f"CAGR: {pf.annualized_return() * 100:+.1f}%")
print(f"Sharpe Ratio: {pf.sharpe_ratio():.2f}")
print(f"Sortino Ratio: {pf.sortino_ratio():.2f}")
print(f"Max Drawdown: {pf.max_drawdown() * 100:.1f}%")
print(f"Win Rate: {pf.trades.win_rate() * 100:.0f}%")
print(f"Profit Factor: {pf.trades.profit_factor():.2f}")
print(f"Total Trades: {pf.trades.count()}")
print(f"Final Value: ${pf.final_value():,.0f}")

In [ ]:
# Compare to buy & hold
print("\n" + "="*60)
print("VS BUY & HOLD")
print("="*60)

bh_return = (close.iloc[-1] / close.iloc[0]) - 1
strategy_return = pf.total_return()

print(f"Strategy Return: {strategy_return * 100:+,.0f}%")
print(f"Buy & Hold Return: {bh_return * 100:+,.0f}%")
print(f"Outperformance: {(strategy_return - bh_return) * 100:+,.0f}%")
print(f"")
print(f"Strategy: $100,000 → ${pf.final_value():,.0f}")
print(f"Buy & Hold: $100,000 → ${100000 * (1 + bh_return):,.0f}")

In [ ]:
# Plot
pf.plot().show()

In [ ]:
# Drawdown plot
pf.drawdowns.plot().show()

In [ ]:
# Trade PnL
pf.trades.plot_pnl().show()